# Join CL SCOTUS Data to SCOTUS Scraped data

The SCOTUS Scraped data is used as a bridge to link the CL SCOTUS opinions to lower court opinions that appealed to SCOTUS. In order to establish the appellate linkage between SCOTUS and lower court opinions, we will leverage the docket number and the date to
1. Link the CL SCOTUS opinions to SCOTUS scraped data
2. Link the CL Lower Court opinions to SCOTUS scraped data

This notebook outlines the first of the two steps above.

# Import Libaries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cl_utils import *

# Load the SCOTUS Scraped data

In [2]:
scotus = pd.read_json("data/scotus_scraped_data.json")
scotus.head()

,docket_number,filename,docket_date,case_title,lower_court,lower_court_case_numbers_raw,lower_court_case_numbers,lower_court_decision_date,lower_court_rehearing_denied_date,year,case_num
0,03-6084,03-6084.htm,2003-08-27,"Ronald Lee Smith, Petitioner v. Larry Reid, Wa...",United States Court of Appeals for the Tenth C...,(03-1016),03-1016,2003-05-16,2003-07-09,3,6084
1,03-10616,03-10616.htm,2004-05-28,"Jimmy Walker, Petitioner v. Florida","District Court of Appeal of Florida, Fourth Di...",(4D02-4272),4D02-4272,2004-04-21,None,3,10616
2,03-777,03-777.htm,2003-11-28,"Willie R. Flint, Petitioner v. ABB Inc., fka A...",United States Court of Appeals for the Elevent...,(02-15029),02-15029,2003-07-21,2003-08-27,3,777
3,03-10170,03-10170.htm,2004-05-05,"Loyda Lugones, Petitioner v. United States",United States Court of Appeals for the Elevent...,(02-12984),02-12984,2004-01-26,None,3,10170
4,03-8917,03-8917.htm,2004-02-19,"Ray Charles Smith, Petitioner v. United States",United States Court of Appeals for the Ninth C...,(03-10003),03-10003,2003-11-13,None,3,8917


## Remove the scotus records that do not have valid lower court info for appellate chain linking

In [3]:
scotus = scotus[(~scotus["lower_court"].isnull()) & (~scotus["lower_court_case_numbers"].isnull())]
len(scotus)

185843

# Load the CL SCOTUS data

The data was downloaded from CL Replica 

In [4]:
cl = pd.read_json("data/cl_scotus_dockets.json")
cl.head()

,docket_id,docket_number,cluster_id,case_name,date_filed,precedential_status,num_authorities
0,7470,290,105849,Securities & Exchange Commission v. Variable A...,1959-03-23,Published,12
1,1269,79-168,110165,"Strycker's Bay Neighborhood Council, Inc. v. K...",1980-01-07,Published,10
2,8948,1047,97897,Detroit United Railway v. City of Detroit,1913-05-26,Published,5
3,9534,13-5380,2642828,Woodward v. Alabama,2013-11-18,Relating-to,15
4,5084,01-8966,119762,Truesdale v. United States,2002-04-15,Published,0


In [65]:
len(cl[cl["num_authorities"] == 0])

430678

In [5]:
len(cl)

499348

In [6]:
cl["cluster_id"].nunique()

499348

# Inspect and remove any empty records for easier processing

In [7]:
cl["docket_id"].nunique()

499155

In [8]:
len(cl)

499348

In [9]:
len(cl[cl["docket_number"].isnull()])

1042

In [10]:
len(cl[cl["docket_number"] == ""])

8501

In [11]:
cl = cl[~cl["docket_number"].isnull()]
cl = cl[cl["docket_number"] != ""]
len(cl)

489805

In [12]:
cl["cluster_id"].nunique()

489805

## Inspect and remove any records with not-Published precedential_status as they are not opinions with precedential value

In [13]:
cl["precedential_status"].value_counts()

precedential_status
Published      489297
Relating-to       493
In-chambers        15
Name: count, dtype: int64

## Inspect number of authorities for each opinion

Most opinions are simply the decision from the court on granting or denying the writ without any substance for precedential value

In [14]:
cl["num_authorities"].describe()

count    489805.000000
mean          0.933382
std           5.725849
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         295.000000
Name: num_authorities, dtype: float64

In [15]:
cl[cl["num_authorities"] == 0]

,docket_id,docket_number,cluster_id,case_name,date_filed,precedential_status,num_authorities
4,5084,01-8966,119762,Truesdale v. United States,2002-04-15,Published,0
7,21360,03-10692,142088,Miller v. United States,2005-01-24,Published,0
10,21957,72-535,108675,United States and Interstate Commerce Commissi...,1973-02-20,Published,0
16,18547,03-7914,136087,"Hoffman v. Jones, Warden",2004-04-26,Published,0
17,15540,02-6511,124689,Dean v. United States,2002-10-21,Published,0
...,...,...,...,...,...,...,...
499337,66728460,No. 19-7937.,9367073,Yaney v. Mason,2020-05-18,Published,0
499341,66728345,No. 19-7979.,9366958,Chadwick v. United States,2020-05-04,Published,0
499343,66728561,No. 19-1021.,9367174,Jessop v. City of Fresno,2020-05-18,Published,0
499346,66728638,No. 19-1238.,9367251,Watson v. McCarthy,2020-05-26,Published,0


In [16]:
cl[cl["num_authorities"] > 0]["num_authorities"].describe()

count    59127.000000
mean         7.732085
std         14.799566
min          1.000000
25%          1.000000
50%          1.000000
75%          8.000000
max        295.000000
Name: num_authorities, dtype: float64

# Clean the CL Docket Numbers & create docket types to filter for only Normal "N" Dockets and Application "A" Dockets to be linked to SCOTUS scraped data

In [17]:
cl["cleaned_docket_number"] = cl["docket_number"].apply(clean_docket_numbers)

In [18]:
cl_exploded = cl.explode("cleaned_docket_number", ignore_index=True)
cl_exploded.head()

,docket_id,docket_number,cluster_id,case_name,date_filed,precedential_status,num_authorities,cleaned_docket_number
0,7470,290,105849,Securities & Exchange Commission v. Variable A...,1959-03-23,Published,12,290
1,1269,79-168,110165,"Strycker's Bay Neighborhood Council, Inc. v. K...",1980-01-07,Published,10,79-168
2,8948,1047,97897,Detroit United Railway v. City of Detroit,1913-05-26,Published,5,1047
3,9534,13-5380,2642828,Woodward v. Alabama,2013-11-18,Relating-to,15,13-5380
4,5084,01-8966,119762,Truesdale v. United States,2002-04-15,Published,0,01-8966


In [19]:
len(cl_exploded)

543525

In [20]:
cl_exploded["cleaned_docket_number"].nunique()

272510

In [21]:
cl_exploded["cluster_id"].nunique()

489805

In [22]:
cl_exploded["year_filed"] = pd.to_datetime(cl_exploded["date_filed"]).dt.year

bins = [-float("inf"), 1998, float("inf")]
labels = ["before_1999", "after_1999"]

cl_exploded["year_bucket"] = pd.cut(cl_exploded["year_filed"], bins=bins, labels=labels)
cl_exploded["year_bucket"].value_counts()

year_bucket
before_1999    328129
after_1999     215396
Name: count, dtype: int64

In [23]:
cl_exploded.drop_duplicates(subset=["year_bucket", "cluster_id"])["year_bucket"].value_counts().sort_index()

year_bucket
before_1999    290138
after_1999     199667
Name: count, dtype: int64

In [24]:
cl_exploded.drop_duplicates(subset=["year_bucket", "cleaned_docket_number"])["year_bucket"].value_counts().sort_index()

year_bucket
before_1999    141865
after_1999     131113
Name: count, dtype: int64

In [25]:
cl_exploded["docket_type"] = cl_exploded["cleaned_docket_number"].apply(get_docket_type)

In [26]:
after_1999 = cl_exploded[cl_exploded["year_bucket"] == "after_1999"]
after_1999["cluster_id"].nunique()

199667

In [27]:
after_1999.drop_duplicates(subset=["cluster_id", "docket_type"])["docket_type"].value_counts().sort_index()

docket_type
A          1988
M          1731
N        194286
O           159
OTHER      2918
R           231
Name: count, dtype: int64

In [28]:
after_1999.drop_duplicates(subset=["cleaned_docket_number", "docket_type"])["docket_type"].value_counts().sort_index()

docket_type
A          1543
M          1956
N        126218
O            32
OTHER      1292
R            72
Name: count, dtype: int64

# Get the normal docket df & join to cl docksts on docket number and date

## Get the docket year from the docket_number for sanity check

In [29]:
cl_subset = cl_exploded[cl_exploded["docket_type"].isin(["N", "A"])].copy()

# Use regex to extract the year based on docket type
cl_subset["docket_year"] = np.select(
    [
        cl_subset["docket_type"] == "N",
        cl_subset["docket_type"] == "A",
    ],
    [
        # Extract digits before hyphen (e.g., "21-123" → "21")
        cl_subset["cleaned_docket_number"].str.extract(r"^(\d+)-")[0],
        # Extract digits before 'A' (e.g., "22A456" → "22")
        cl_subset["cleaned_docket_number"].str.extract(r"^(\d+)A")[0],
    ],
    default=np.nan
)

# Convert to Int64, allowing for blanks (NaN)
cl_subset["docket_year"] = pd.to_numeric(cl_subset["docket_year"], errors="coerce").astype("Int64")

cl_subset.head()

,docket_id,docket_number,cluster_id,case_name,date_filed,precedential_status,num_authorities,cleaned_docket_number,year_filed,year_bucket,docket_type,docket_year
0,7470,290,105849,Securities & Exchange Commission v. Variable A...,1959-03-23,Published,12,290,1959,before_1999,N,<NA>
1,1269,79-168,110165,"Strycker's Bay Neighborhood Council, Inc. v. K...",1980-01-07,Published,10,79-168,1980,before_1999,N,79
2,8948,1047,97897,Detroit United Railway v. City of Detroit,1913-05-26,Published,5,1047,1913,before_1999,N,<NA>
3,9534,13-5380,2642828,Woodward v. Alabama,2013-11-18,Relating-to,15,13-5380,2013,after_1999,N,13
4,5084,01-8966,119762,Truesdale v. United States,2002-04-15,Published,0,01-8966,2002,after_1999,N,1


In [30]:
len(cl_subset)

523079

In [31]:
cl_subset["cluster_id"].nunique()

473102

In [32]:
cl_subset["cleaned_docket_number"].nunique()

264072

## Remove any records with docket_year before 1999 or docket_year after 2024 as they do not exist in scotus data

In [33]:
cl_subset = cl_subset[(cl_subset["docket_year"] >= 0) & (cl_subset["docket_year"] <= 25) | (cl_subset["docket_year"] == 99)]

In [34]:
len(cl_subset)

199445

In [35]:
cl_subset["cluster_id"].nunique()

186155

In [36]:
cl_subset["cleaned_docket_number"].nunique()

122669

In [37]:
cl_subset["year_bucket"].value_counts()

year_bucket
after_1999     199427
before_1999        18
Name: count, dtype: int64

## Remove any records with year_bucket before_1999 as they do not exist in scotus data

In [38]:
cl_subset[cl_subset["year_bucket"] == "before_1999"]["year_filed"].value_counts().sort_index()

year_filed
1927    1
1931    1
1933    1
1934    1
1941    1
1943    1
1956    1
1957    1
1959    1
1960    3
1963    1
1964    2
1966    2
1979    1
Name: count, dtype: int64

In [39]:
cl_subset = cl_subset[cl_subset["year_bucket"] == "after_1999"]

In [40]:
len(cl_subset)

199427

In [41]:
cl_subset["cluster_id"].nunique()

186137

In [42]:
cl_subset["cleaned_docket_number"].nunique()

122664

In [43]:
cl_subset.drop_duplicates(subset=["cluster_id", "docket_type"])["docket_type"].value_counts().sort_index()

docket_type
A      1987
N    185347
Name: count, dtype: int64

In [44]:
cl_subset.drop_duplicates(subset=["cleaned_docket_number", "docket_type"])["docket_type"].value_counts().sort_index()

docket_type
A      1542
N    121122
Name: count, dtype: int64

## Sanity check for dockets that exist in CL but not in SCOTUS scraped data

- There are generally a lot more data in CL because of duplicates and the possibilities that one docket number could have multiple opinions.
- There are many missing dockets for year 1999 and 2000, this is because SCOTUS data become available beginning 2000, but not all the data are available from the website.
- For remaining years, I picked a few for sanity check and confirmed these are dockets that cannot be found from the SCOTUS docket search, these will need to be analyzed through parsing.

In [45]:
missing_dockets = {}
valid_dockets = {}

for docket_type in ["N", "A"]:
    print("*****", docket_type, "*****")
    cl_subset_subset = cl_subset[cl_subset["docket_type"] == docket_type]
    missing_dockets[docket_type] = {}
    valid_dockets[docket_type] = {}
    
    for yr in list(range(0, 25)) + [99]:
        print("---", yr, "---")
        
        cl_yr = cl_subset_subset[cl_subset_subset["docket_year"] == yr]
        print("num_cl: ", len(cl_yr))
        
        scotus_yr = scotus[scotus["year"] == yr]
        print("num_scotus: ", len(scotus_yr))
    
        missing_dockets[docket_type][yr] = list(set(cl_yr["cleaned_docket_number"].to_list()) - set(scotus_yr["docket_number"].to_list()))
        valid_dockets[docket_type][yr] = cl_yr[cl_yr["cleaned_docket_number"].isin(scotus_yr["docket_number"].to_list())]["cleaned_docket_number"].to_list()
        
        print("docket nums in cl not in scotus: ", len(missing_dockets[docket_type][yr]))
        print("docket nums in cl and in scotus: ", len(valid_dockets[docket_type][yr]))

    print("==========================================================")
    print(f"number of dockets from {docket_type} dockets: ", len(set(cl_subset_subset["cleaned_docket_number"].to_list())))
    print(f"number of dockets from {docket_type} dockets in valid years: ", len(set([item for sublist in valid_dockets[docket_type].values() for item in sublist])) + len(set([item for sublist in missing_dockets[docket_type].values() for item in sublist])))
    print(f"number of valid dockets from {docket_type} dockets: ", len(set([item for sublist in valid_dockets[docket_type].values() for item in sublist])))
    print(f"number of missing dockets from {docket_type} dockets: ", len(set([item for sublist in missing_dockets[docket_type].values() for item in sublist])))

***** N *****
--- 0 ---
num_cl:  12166
num_scotus:  1675
docket nums in cl not in scotus:  6165
docket nums in cl and in scotus:  2929
--- 1 ---
num_cl:  16357
num_scotus:  7682
docket nums in cl not in scotus:  239
docket nums in cl and in scotus:  15895
--- 2 ---
num_cl:  21363
num_scotus:  8263
docket nums in cl not in scotus:  242
docket nums in cl and in scotus:  20754
--- 3 ---
num_cl:  21677
num_scotus:  8621
docket nums in cl not in scotus:  235
docket nums in cl and in scotus:  21014
--- 4 ---
num_cl:  16998
num_scotus:  8326
docket nums in cl not in scotus:  236
docket nums in cl and in scotus:  16457
--- 5 ---
num_cl:  4976
num_scotus:  9462
docket nums in cl not in scotus:  159
docket nums in cl and in scotus:  4803
--- 6 ---
num_cl:  369
num_scotus:  9811
docket nums in cl not in scotus:  2
docket nums in cl and in scotus:  367
--- 7 ---
num_cl:  198
num_scotus:  8960
docket nums in cl not in scotus:  0
docket nums in cl and in scotus:  198
--- 8 ---
num_cl:  309
num_scotu

In [46]:
for key, val in missing_dockets["N"].items():
    print("year: ", key, "num missing: ", len(val))

year:  0 num missing:  6165
year:  1 num missing:  239
year:  2 num missing:  242
year:  3 num missing:  235
year:  4 num missing:  236
year:  5 num missing:  159
year:  6 num missing:  2
year:  7 num missing:  0
year:  8 num missing:  2
year:  9 num missing:  121
year:  10 num missing:  225
year:  11 num missing:  220
year:  12 num missing:  204
year:  13 num missing:  227
year:  14 num missing:  204
year:  15 num missing:  214
year:  16 num missing:  209
year:  17 num missing:  114
year:  18 num missing:  194
year:  19 num missing:  157
year:  20 num missing:  0
year:  21 num missing:  1
year:  22 num missing:  1
year:  23 num missing:  0
year:  24 num missing:  0
year:  99 num missing:  7358


In [47]:
for key, val in missing_dockets["A"].items():
    print("year: ", key, "num missing: ", len(val))

year:  0 num missing:  98
year:  1 num missing:  99
year:  2 num missing:  102
year:  3 num missing:  13
year:  4 num missing:  5
year:  5 num missing:  14
year:  6 num missing:  1
year:  7 num missing:  0
year:  8 num missing:  0
year:  9 num missing:  4
year:  10 num missing:  5
year:  11 num missing:  7
year:  12 num missing:  3
year:  13 num missing:  5
year:  14 num missing:  2
year:  15 num missing:  6
year:  16 num missing:  1
year:  17 num missing:  3
year:  18 num missing:  6
year:  19 num missing:  5
year:  20 num missing:  0
year:  21 num missing:  0
year:  22 num missing:  0
year:  23 num missing:  0
year:  24 num missing:  0
year:  99 num missing:  105


## Join the scotus scraped normal dockets to cl dockets 
1. on docket number
2. cl date_filed must be within 2-year of scotus docket_date

In [48]:
scotus['docket_date'] = pd.to_datetime(scotus['docket_date'])
cl_subset['date_filed'] = pd.to_datetime(cl_subset['date_filed'])

cl_subset = cl_subset.add_prefix("cl_")
scotus = scotus.add_prefix("scotus_")

merged = pd.merge(cl_subset, scotus, left_on='cl_cleaned_docket_number', right_on='scotus_docket_number', how='left')
merged = merged[~merged["scotus_docket_number"].isnull()]
merged.head()

,cl_docket_id,cl_docket_number,cl_cluster_id,cl_case_name,cl_date_filed,cl_precedential_status,cl_num_authorities,cl_cleaned_docket_number,cl_year_filed,cl_year_bucket,...,scotus_filename,scotus_docket_date,scotus_case_title,scotus_lower_court,scotus_lower_court_case_numbers_raw,scotus_lower_court_case_numbers,scotus_lower_court_decision_date,scotus_lower_court_rehearing_denied_date,scotus_year,scotus_case_num
0,9534,13-5380,2642828,Woodward v. Alabama,2013-11-18,Relating-to,15,13-5380,2013,after_1999,...,13-5380.htm,2013-07-18,"Mario Dion Woodward, Petitioner v. Alabama",Court of Criminal Appeals of Alabama,(CR-08-0145),CR-08-0145,2011-12-16,2012-08-24,13.0,5380.0
1,5084,01-8966,119762,Truesdale v. United States,2002-04-15,Published,0,01-8966,2002,after_1999,...,01-8966.htm,2002-03-13,"Alvin B. Truesdale, Petitioner v. United States",United States Court of Appeals for the Fourth ...,(01-7004),01-7004,None,None,1.0,8966.0
2,21360,03-10692,142088,Miller v. United States,2005-01-24,Published,0,03-10692,2005,after_1999,...,03-10692.htm,2004-06-04,"Alton Miller, Petitioner v. United States",United States Court of Appeals for the Elevent...,(03-16031-I),03-16031-I,2004-04-08,None,3.0,10692.0
3,14214,01-9027,120409,"Anderson v. Mayle, Warden",2002-05-13,Published,1,01-9027,2002,after_1999,...,01-9027.htm,2002-03-15,"Tony Anderson, Petitioner v. Denise Mayle, War...",United States Court of Appeals for the Ninth C...,(00-55684),00-55684,None,None,1.0,9027.0
4,2614,03-10458,138199,"Gavaldon v. Cambra, Warden",2004-10-04,Published,1,03-10458,2004,after_1999,...,03-10458.htm,2004-05-20,"David Gavaldon, Petitioner v. Steven J. Cambr...",United States Court of Appeals for the Ninth C...,(02-17061),02-17061,2004-02-26,None,3.0,10458.0


In [49]:
merged.drop_duplicates(subset=["cl_cluster_id", "cl_docket_type"])["cl_docket_type"].value_counts().sort_index()

cl_docket_type
A      1297
N    163323
Name: count, dtype: int64

In [50]:
merged.drop_duplicates(subset=["cl_cleaned_docket_number", "cl_docket_type"])["cl_docket_type"].value_counts().sort_index()

cl_docket_type
A      1057
N    104393
Name: count, dtype: int64

In [51]:
# Filter for date_filed within 2 year of docket_date
# i.e., docket_date - 365 * 2 <= date_filed <= docket_date + 365 * 2
mask = (merged['cl_date_filed'] >= merged['scotus_docket_date'] - pd.Timedelta(days=365*2)) & \
       (merged['cl_date_filed'] <= merged['scotus_docket_date'] + pd.Timedelta(days=365*2))

result = merged[mask]
result.head()

,cl_docket_id,cl_docket_number,cl_cluster_id,cl_case_name,cl_date_filed,cl_precedential_status,cl_num_authorities,cl_cleaned_docket_number,cl_year_filed,cl_year_bucket,...,scotus_filename,scotus_docket_date,scotus_case_title,scotus_lower_court,scotus_lower_court_case_numbers_raw,scotus_lower_court_case_numbers,scotus_lower_court_decision_date,scotus_lower_court_rehearing_denied_date,scotus_year,scotus_case_num
0,9534,13-5380,2642828,Woodward v. Alabama,2013-11-18,Relating-to,15,13-5380,2013,after_1999,...,13-5380.htm,2013-07-18,"Mario Dion Woodward, Petitioner v. Alabama",Court of Criminal Appeals of Alabama,(CR-08-0145),CR-08-0145,2011-12-16,2012-08-24,13.0,5380.0
1,5084,01-8966,119762,Truesdale v. United States,2002-04-15,Published,0,01-8966,2002,after_1999,...,01-8966.htm,2002-03-13,"Alvin B. Truesdale, Petitioner v. United States",United States Court of Appeals for the Fourth ...,(01-7004),01-7004,None,None,1.0,8966.0
2,21360,03-10692,142088,Miller v. United States,2005-01-24,Published,0,03-10692,2005,after_1999,...,03-10692.htm,2004-06-04,"Alton Miller, Petitioner v. United States",United States Court of Appeals for the Elevent...,(03-16031-I),03-16031-I,2004-04-08,None,3.0,10692.0
3,14214,01-9027,120409,"Anderson v. Mayle, Warden",2002-05-13,Published,1,01-9027,2002,after_1999,...,01-9027.htm,2002-03-15,"Tony Anderson, Petitioner v. Denise Mayle, War...",United States Court of Appeals for the Ninth C...,(00-55684),00-55684,None,None,1.0,9027.0
4,2614,03-10458,138199,"Gavaldon v. Cambra, Warden",2004-10-04,Published,1,03-10458,2004,after_1999,...,03-10458.htm,2004-05-20,"David Gavaldon, Petitioner v. Steven J. Cambr...",United States Court of Appeals for the Ninth C...,(02-17061),02-17061,2004-02-26,None,3.0,10458.0


## Inspect the joined results

In [52]:
result["cl_cleaned_docket_number"].nunique()

104771

In [53]:
result["cl_cluster_id"].nunique()

163672

In [54]:
joineable_dockets = list(result["cl_cleaned_docket_number"].unique())
cl_exploded[cl_exploded["cleaned_docket_number"].isin(joineable_dockets)]["cluster_id"].nunique()

163727

## Check for valid dockets that could be joined but did not due to date mismatch

In [55]:
valid_docket_nums = [
    item
    for inner_dict in valid_dockets.values()
    for sublist in inner_dict.values()
    for item in sublist
]
len(valid_docket_nums)

172958

In [56]:
# confirm only valid dockets were joined as expected
assert len(set(result["cl_cleaned_docket_number"].to_list()) - set(valid_docket_nums)) == 0

In [57]:
set(valid_docket_nums) - set(result["cl_cleaned_docket_number"].to_list())

{'00-1295',
 '02A1001',
 '02A1005',
 '02A1048',
 '02A1060',
 '02A1067',
 '02A1073',
 '02A1086',
 '02A724',
 '02A952',
 '03A1002',
 '03A1011',
 '03A1020',
 '03A1023',
 '03A1031',
 '03A1034',
 '03A1056',
 '03A1057',
 '03A1058',
 '03A111',
 '03A113',
 '03A14',
 '03A142',
 '03A185',
 '03A202',
 '03A222',
 '03A224',
 '03A23',
 '03A266',
 '03A27',
 '03A283',
 '03A284',
 '03A287',
 '03A292',
 '03A308',
 '03A324',
 '03A36',
 '03A373',
 '03A375',
 '03A38',
 '03A390',
 '03A393',
 '03A400',
 '03A448',
 '03A463',
 '03A48',
 '03A490',
 '03A497',
 '03A511',
 '03A513',
 '03A542',
 '03A554',
 '03A563',
 '03A576',
 '03A581',
 '03A584',
 '03A590',
 '03A594',
 '03A60',
 '03A606',
 '03A616',
 '03A62',
 '03A630',
 '03A633',
 '03A637',
 '03A650',
 '03A653',
 '03A662',
 '03A663',
 '03A666',
 '03A673',
 '03A687',
 '03A69',
 '03A697',
 '03A698',
 '03A706',
 '03A729',
 '03A733',
 '03A742',
 '03A744',
 '03A76',
 '03A773',
 '03A797',
 '03A799',
 '03A812',
 '03A815',
 '03A826',
 '03A827',
 '03A838',
 '03A873',
 '0

In [58]:
len(set(valid_docket_nums) - set(result["cl_cleaned_docket_number"].to_list()))

679

## Inspect a few examples that did not get joined, these are the ones that require manual review or parsing

In [59]:
merged[merged["cl_cleaned_docket_number"] == "00-1295"]

,cl_docket_id,cl_docket_number,cl_cluster_id,cl_case_name,cl_date_filed,cl_precedential_status,cl_num_authorities,cl_cleaned_docket_number,cl_year_filed,cl_year_bucket,...,scotus_filename,scotus_docket_date,scotus_case_title,scotus_lower_court,scotus_lower_court_case_numbers_raw,scotus_lower_court_case_numbers,scotus_lower_court_decision_date,scotus_lower_court_rehearing_denied_date,scotus_year,scotus_case_num
233,1801,00-1295,128673,"Community Health Partners, Inc. v. Kentucky",2003-04-07,Published,0,00-1295,2003,after_1999,...,00-1295.htm,2001-02-14,"Community Health Partners, Inc., and Reservoir...",United States Court of Appeals for the Sixth C...,(98-5941),98-5941,None,None,0.0,1295.0
104671,66474189,No. 00-1295,9207871,"Community Health Partners, Inc. v. Kentucky",2003-04-07,Published,0,00-1295,2003,after_1999,...,00-1295.htm,2001-02-14,"Community Health Partners, Inc., and Reservoir...",United States Court of Appeals for the Sixth C...,(98-5941),98-5941,None,None,0.0,1295.0
105127,66474188,No. 00-1295,9207870,"Community Health Partners, Inc. v. Kentucky",2003-04-07,Published,0,00-1295,2003,after_1999,...,00-1295.htm,2001-02-14,"Community Health Partners, Inc., and Reservoir...",United States Court of Appeals for the Sixth C...,(98-5941),98-5941,None,None,0.0,1295.0


In [60]:
merged[merged["cl_cleaned_docket_number"] == "20-1199"]

,cl_docket_id,cl_docket_number,cl_cluster_id,cl_case_name,cl_date_filed,cl_precedential_status,cl_num_authorities,cl_cleaned_docket_number,cl_year_filed,cl_year_bucket,...,scotus_filename,scotus_docket_date,scotus_case_title,scotus_lower_court,scotus_lower_court_case_numbers_raw,scotus_lower_court_case_numbers,scotus_lower_court_decision_date,scotus_lower_court_rehearing_denied_date,scotus_year,scotus_case_num
199299,67538851,20-1199,10049657,"Students for Fair Admissions, Inc. v. Presiden...",2023-06-29,Published,99,20-1199,2023,after_1999,...,20-1199.json,2021-03-01,"Students for Fair Admissions, Inc., Petitioner...",United States Court of Appeals for the First C...,(19-2005),19-2005,2020-11-12,None,20.0,1199.0
199300,67538851,20-1199,9410338,"Students for Fair Admissions, Inc. v. Presiden...",2023-06-29,Published,103,20-1199,2023,after_1999,...,20-1199.json,2021-03-01,"Students for Fair Admissions, Inc., Petitioner...",United States Court of Appeals for the First C...,(19-2005),19-2005,2020-11-12,None,20.0,1199.0


In [61]:
merged[merged["cl_cleaned_docket_number"] == "20A136"]

,cl_docket_id,cl_docket_number,cl_cluster_id,cl_case_name,cl_date_filed,cl_precedential_status,cl_num_authorities,cl_cleaned_docket_number,cl_year_filed,cl_year_bucket,...,scotus_filename,scotus_docket_date,scotus_case_title,scotus_lower_court,scotus_lower_court_case_numbers_raw,scotus_lower_court_case_numbers,scotus_lower_court_decision_date,scotus_lower_court_rehearing_denied_date,scotus_year,scotus_case_num
29013,59240167,20A136,4855229,South Bay United Pentecostal Church v. Newsom,2021-02-05,Relating-to,4,20A136,2021,after_1999,...,20A136.json,NaT,"South Bay United Pentecostal Church, et al., A...",United States Court of Appeals for the Ninth C...,(20-55533),20-55533,None,None,20.0,136.0


In [69]:
parse = cl[~cl["cluster_id"].isin(list(set(result["cl_cluster_id"].to_list())))]
parse["cluster_id"].nunique()

326133

In [70]:
len(parse)

326133

In [72]:
## These are ones that need to be parsed by LLM
len(parse) - len(parse[parse["num_authorities"] == 0])

33673

# Save the results for next step join

In [62]:
result.columns

Index(['cl_docket_id', 'cl_docket_number', 'cl_cluster_id', 'cl_case_name',
       'cl_date_filed', 'cl_precedential_status', 'cl_num_authorities',
       'cl_cleaned_docket_number', 'cl_year_filed', 'cl_year_bucket',
       'cl_docket_type', 'cl_docket_year', 'scotus_docket_number',
       'scotus_filename', 'scotus_docket_date', 'scotus_case_title',
       'scotus_lower_court', 'scotus_lower_court_case_numbers_raw',
       'scotus_lower_court_case_numbers', 'scotus_lower_court_decision_date',
       'scotus_lower_court_rehearing_denied_date', 'scotus_year',
       'scotus_case_num'],
      dtype='object')

In [63]:
result.to_csv("data/joined_scotus.csv", index=False)